# LeetCode 2001 — Number of Pairs of Interchangeable Rectangles bằng Python

Link đề: https://leetcode.com/problems/number-of-pairs-of-interchangeable-rectangles/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Math, Counting, Number Theory

Skeleton LeetCode:

```python
class Solution:
    def interchangeableRectangles(self, rectangles: List[List[int]]) -> int:
        ...
```

## 1. Mục tiêu học

1. So sánh hai phân số **chính xác** bằng **nhân chéo** (`a·d == b·c`) và hiểu vì sao không nên tin hoàn toàn vào số thực.
2. Chuẩn hóa phân số bằng **ước chung lớn nhất** (`math.gcd`) để mỗi tỉ lệ có **một khóa duy nhất**.
3. Dùng **bảng băm đếm tần suất** (`dict`, `Counter`) để đếm số cặp trong `O(n)` thay vì `O(n²)`.
4. Nắm hai cách đếm cặp: công thức tổ hợp `C(c, 2) = c(c−1)/2` và kỹ thuật **"cộng số đã thấy trước đó"**.
5. Kiểm thử đối chiếu brute force với các cách tối ưu trên dữ liệu ngẫu nhiên có seed cố định.

## 2. Đề bài & ví dụ

Cho `n` hình chữ nhật, `rectangles[i] = [width_i, height_i]`. Hai hình `i < j` gọi là **hoán đổi được** nếu chúng có **cùng tỉ lệ** rộng/cao: `width_i / height_i == width_j / height_j` (chia số thực, không phải chia nguyên).

Trả về **số cặp** `(i, j)` hoán đổi được.

**Ví dụ 1:** `[[4,8],[3,6],[10,20],[15,30]]` → `6`. Cả 4 hình đều có tỉ lệ `1/2`, nên mọi cặp đều hợp lệ: `C(4,2) = 6`.

**Ví dụ 2:** `[[4,5],[7,8]]` → `0`. Tỉ lệ `0.8` và `0.875` khác nhau.

**Ràng buộc:** `1 ≤ n ≤ 10⁵`, `1 ≤ width_i, height_i ≤ 10⁵`.

In [1]:
import random
import time
from math import gcd, comb
from collections import Counter, defaultdict
from fractions import Fraction
from typing import List

examples = [([[4, 8], [3, 6], [10, 20], [15, 30]], 6), ([[4, 5], [7, 8]], 0)]
for rects, exp in examples:
    print(rects, "-> kỳ vọng", exp)

[[4, 8], [3, 6], [10, 20], [15, 30]] -> kỳ vọng 6
[[4, 5], [7, 8]] -> kỳ vọng 0


## 3. Kiến thức nền

### 3.1 So sánh phân số bằng nhân chéo

Với `b, d > 0`: `a/b == c/d` ⇔ `a·d == b·c`. Phép nhân số nguyên là **chính xác tuyệt đối** (Python không tràn số), nên đây là cách so sánh an toàn nhất.

In [2]:
def same_ratio(r1, r2):
    (a, b), (c, d) = r1, r2
    return a * d == b * c

print(same_ratio([4, 8], [3, 6]), same_ratio([4, 5], [7, 8]))

True False


### 3.2 `math.gcd` và phân số tối giản

`gcd(a, b)` = ước chung lớn nhất. Chia cả tử và mẫu cho `gcd` ta được **phân số tối giản** — hai phân số bằng nhau ⇔ dạng tối giản **giống hệt nhau**. Vậy cặp `(a // g, b // g)` là một **khóa** tốt cho bảng băm.

In [3]:
for w, h in [[4, 8], [3, 6], [10, 20], [15, 30], [4, 5]]:
    g = gcd(w, h)
    print(f"{w}/{h}: gcd={g} -> tối giản ({w // g}, {h // g})")

4/8: gcd=4 -> tối giản (1, 2)
3/6: gcd=3 -> tối giản (1, 2)
10/20: gcd=10 -> tối giản (1, 2)
15/30: gcd=15 -> tối giản (1, 2)
4/5: gcd=1 -> tối giản (4, 5)


**Thuật toán Euclid bên trong `gcd`.** `gcd(a, b) = gcd(b, a mod b)`, dừng khi `b = 0`. Mỗi hai bước, số lớn giảm ít nhất một nửa ⇒ chỉ `O(log min(a, b))` bước. Tự cài và in từng bước để thấy nó nhanh thế nào:

In [4]:
def gcd_trace(a, b):
    steps = 0
    while b:
        print(f"  gcd({a}, {b}) -> gcd({b}, {a % b})")
        a, b = b, a % b
        steps += 1
    print(f"  = {a} sau {steps} bước")
    return a

gcd_trace(15, 30)
gcd_trace(99999, 100000)

  gcd(15, 30) -> gcd(30, 15)
  gcd(30, 15) -> gcd(15, 0)
  = 15 sau 2 bước
  gcd(99999, 100000) -> gcd(100000, 99999)
  gcd(100000, 99999) -> gcd(99999, 1)
  gcd(99999, 1) -> gcd(1, 0)
  = 1 sau 3 bước


1

### 3.3 Đếm tần suất bằng `dict` / `Counter`

`Counter(iterable)` đếm số lần xuất hiện của mỗi phần tử. Với `dict` thường: `cnt[key] = cnt.get(key, 0) + 1`. Khóa phải **hashable** — `tuple` thì được, `list` thì không.

In [5]:
keys = [(1, 2), (1, 2), (4, 5), (1, 2)]
print(Counter(keys))
cnt = {}
for k in keys:
    cnt[k] = cnt.get(k, 0) + 1
print(cnt)
print("tuple hash được:", (1, 2).__hash__ is not None, "| list hash được:", [].__hash__ is not None)

Counter({(1, 2): 3, (4, 5): 1})
{(1, 2): 3, (4, 5): 1}
tuple hash được: True | list hash được: False


### 3.4 Đếm số cặp từ tần suất

Nếu một nhóm có `c` phần tử thì số cặp `(i < j)` trong nhóm là `C(c, 2) = c·(c−1)/2`. Cách thứ hai: duyệt lần lượt, mỗi phần tử mới tạo cặp với **tất cả** phần tử cùng nhóm **đã gặp trước** nó ⇒ `ans += cnt[key]` rồi mới `cnt[key] += 1`. Hai cách cho cùng kết quả vì `0 + 1 + ... + (c−1) = c(c−1)/2`.

In [6]:
c = 4
print("C(4,2) =", comb(c, 2), "=", c * (c - 1) // 2, "=", sum(range(c)))

C(4,2) = 6 = 6 = 6


### 3.5 `Fraction` — phân số chính xác của Python

`Fraction(4, 8)` tự tối giản thành `1/2` và hash được, nên cũng dùng làm khóa được. Tiện nhưng chậm hơn tuple `(a//g, b//g)` vài lần.

In [7]:
print(Fraction(4, 8), Fraction(3, 6) == Fraction(15, 30), hash(Fraction(1, 2)) == hash(Fraction(10, 20)))

1/2 True True


## 4. Ý tưởng nền tảng / trực giác

Tính tay một ví dụ trộn: `[[2,3],[4,6],[1,1],[6,9],[5,5],[3,2]]`.

| i | (w, h) | tối giản |
|---|---|---|
| 0 | (2, 3) | (2, 3) |
| 1 | (4, 6) | (2, 3) |
| 2 | (1, 1) | (1, 1) |
| 3 | (6, 9) | (2, 3) |
| 4 | (5, 5) | (1, 1) |
| 5 | (3, 2) | (3, 2) |

Nhóm `(2,3)` có 3 hình ⇒ `3` cặp; nhóm `(1,1)` có 2 hình ⇒ `1` cặp; nhóm `(3,2)` có 1 hình ⇒ `0`. Tổng `4`. Chú ý `(2,3)` và `(3,2)` **khác** nhau (xoay hình không được tính).

Bài toán thực chất là: **gom các phần tử bằng nhau (theo tỉ lệ) rồi đếm cặp trong từng nhóm.**

In [8]:
mixed = [[2, 3], [4, 6], [1, 1], [6, 9], [5, 5], [3, 2]]
groups = Counter((w // gcd(w, h), h // gcd(w, h)) for w, h in mixed)
print(groups, "-> tổng số cặp:", sum(c * (c - 1) // 2 for c in groups.values()))

Counter({(2, 3): 3, (1, 1): 2, (3, 2): 1}) -> tổng số cặp: 4


## 5. Cách 1 — Brute Force / baseline: xét mọi cặp

**Ý tưởng:** hai vòng lặp `i < j`, dùng nhân chéo để kiểm tra cùng tỉ lệ.

In [9]:
def count_brute(rectangles):
    n = len(rectangles)
    ans = 0
    for i in range(n):
        wi, hi = rectangles[i]
        for j in range(i + 1, n):
            wj, hj = rectangles[j]
            if wi * hj == wj * hi:
                ans += 1
    return ans

for rects, exp in examples:
    got = count_brute(rects); print(got, "kỳ vọng", exp); assert got == exp
print("ví dụ trộn:", count_brute(mixed))

6 kỳ vọng 6
0 kỳ vọng 0
ví dụ trộn: 4


**Độ phức tạp:** Time `O(n²)`, Space `O(1)`. Với `n = 10⁵` cần ~`5·10⁹` phép so sánh ⇒ **TLE**.

**Nút thắt:** ta so sánh từng cặp một, trong khi "cùng tỉ lệ" là một **quan hệ tương đương** — có thể gom nhóm một lần rồi đếm.

In [10]:
rng = random.Random(0)
for n in (500, 1000, 2000):
    data = [[rng.randint(1, 20), rng.randint(1, 20)] for _ in range(n)]
    t = time.perf_counter(); count_brute(data)
    print(f"n={n}: {time.perf_counter() - t:.3f}s  (gấp đôi n -> thời gian ~x4)")

n=500: 0.004s  (gấp đôi n -> thời gian ~x4)
n=1000: 0.017s  (gấp đôi n -> thời gian ~x4)
n=2000: 0.065s  (gấp đôi n -> thời gian ~x4)


## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** "cùng tỉ lệ" là quan hệ tương đương (phản xạ, đối xứng, bắc cầu). Mỗi lớp tương đương được đại diện bởi **một phân số tối giản duy nhất** `(w/g, h/g)` với `g = gcd(w, h)`.

**Quy tắc "nếu… thì…":**
- **Nếu** hai hình có cùng khóa tối giản **thì** chúng hoán đổi được, và ngược lại.
- **Nếu** một nhóm có `c` hình **thì** nhóm đó đóng góp `c(c−1)/2` cặp.
- **Nếu** duyệt từ trái sang phải **thì** hình thứ `j` tạo đúng `cnt[key]` cặp mới với các hình trước nó cùng nhóm.

**Invariant (cách đếm trực tuyến):** sau khi xử lý `j` hình đầu, `ans` = số cặp hợp lệ `(i < i')` với `i' < j`, và `cnt[key]` = số hình trong `j` hình đầu có khóa `key`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Khóa là số thực `w / h`

**Giải thích:** dùng `w / h` (float) làm khóa của `Counter`. Với ràng buộc `≤ 10⁵`, hai phân số khác nhau chênh nhau ít nhất `1/(10⁵·10⁵) = 10⁻¹⁰`, còn sai số làm tròn của `float` (double) chỉ cỡ `10⁻¹⁶` lần giá trị ⇒ không bị nhầm. Nhưng đây là lập luận **phụ thuộc ràng buộc** — ra ngoài phạm vi có thể sai (xem mục 13).

In [11]:
def count_float(rectangles):
    cnt = Counter(w / h for w, h in rectangles)
    return sum(c * (c - 1) // 2 for c in cnt.values())

for rects, exp in examples:
    assert count_float(rects) == exp
print([count_float(r) for r, _ in examples], "| ví dụ trộn:", count_float(mixed))

[6, 0] | ví dụ trộn: 4


In [12]:
class SolutionFloat:
    def interchangeableRectangles(self, rectangles: List[List[int]]) -> int:
        cnt = Counter(w / h for w, h in rectangles)
        return sum(c * (c - 1) // 2 for c in cnt.values())

print([SolutionFloat().interchangeableRectangles(r) for r, _ in examples])

[6, 0]


**Độ phức tạp Cách 2:** Time `O(n)` trung bình (băm), Space `O(n)`.

### 7.2 Cách 3 — Khóa là phân số tối giản `(w/g, h/g)` (chính xác tuyệt đối)

**Giải thích từng đoạn:**
1. Với mỗi hình, `g = gcd(w, h)`, khóa `(w // g, h // g)`.
2. `ans += cnt[key]` — hình hiện tại ghép cặp với mọi hình cùng khóa đã gặp.
3. `cnt[key] += 1`.

In [13]:
def count_gcd(rectangles):
    cnt = defaultdict(int)
    ans = 0
    for w, h in rectangles:
        g = gcd(w, h)
        key = (w // g, h // g)
        ans += cnt[key]
        cnt[key] += 1
    return ans

for rects, exp in examples:
    assert count_gcd(rects) == exp
print([count_gcd(r) for r, _ in examples], "| ví dụ trộn:", count_gcd(mixed))

[6, 0] | ví dụ trộn: 4


In [14]:
class SolutionGcd:
    def interchangeableRectangles(self, rectangles: List[List[int]]) -> int:
        cnt = defaultdict(int)
        ans = 0
        for w, h in rectangles:
            g = gcd(w, h)
            key = (w // g, h // g)
            ans += cnt[key]
            cnt[key] += 1
        return ans

print([SolutionGcd().interchangeableRectangles(r) for r, _ in examples])

[6, 0]


**Độ phức tạp Cách 3:** mỗi `gcd` tốn `O(log V)` (thuật toán Euclid, `V = 10⁵`) ⇒ Time `O(n log V)`, Space `O(n)`.

### 7.3 Cách 4 — Sắp xếp theo khóa rồi đếm các đoạn bằng nhau

Không dùng băm: tạo danh sách khóa tối giản, **sắp xếp**, các khóa bằng nhau nằm liền nhau; mỗi đoạn dài `c` cho `c(c−1)/2` cặp. Hữu ích khi không muốn/không được dùng hash map.

In [15]:
def count_sort(rectangles):
    keys = sorted((w // gcd(w, h), h // gcd(w, h)) for w, h in rectangles)
    ans, i, n = 0, 0, len(keys)
    while i < n:
        j = i
        while j < n and keys[j] == keys[i]:
            j += 1
        c = j - i
        ans += c * (c - 1) // 2
        i = j
    return ans

for rects, exp in examples:
    assert count_sort(rects) == exp
print([count_sort(r) for r, _ in examples], "| ví dụ trộn:", count_sort(mixed))

[6, 0] | ví dụ trộn: 4


In [16]:
class SolutionSort:
    def interchangeableRectangles(self, rectangles: List[List[int]]) -> int:
        keys = sorted((w // gcd(w, h), h // gcd(w, h)) for w, h in rectangles)
        ans = run = 0
        for i in range(len(keys)):
            run = run + 1 if i > 0 and keys[i] == keys[i - 1] else 1
            ans += run - 1           # phần tử thứ run trong đoạn ghép với run-1 phần tử trước
        return ans

print([SolutionSort().interchangeableRectangles(r) for r, _ in examples])

[6, 0]


**Độ phức tạp Cách 4:** Time `O(n log n)` (sắp xếp), Space `O(n)`.

### 7.4 Cách 5 — Khóa `Fraction`

Ngắn gọn nhất và chính xác, đổi lại chậm hơn do mỗi `Fraction` là một đối tượng Python có gcd bên trong.

In [17]:
def count_fraction(rectangles):
    cnt = Counter(Fraction(w, h) for w, h in rectangles)
    return sum(comb(c, 2) for c in cnt.values())

print([count_fraction(r) for r, _ in examples], "| ví dụ trộn:", count_fraction(mixed))

[6, 0] | ví dụ trộn: 4


## 8. TRACE từng bước

Trace Cách 3 trên ví dụ trộn `[[2,3],[4,6],[1,1],[6,9],[5,5],[3,2]]`:

| j | (w, h) | g | khóa | cnt[khóa] trước | ans sau |
|---|---|---|---|---|---|
| 0 | (2,3) | 1 | (2,3) | 0 | 0 |
| 1 | (4,6) | 2 | (2,3) | 1 | 1 |
| 2 | (1,1) | 1 | (1,1) | 0 | 1 |
| 3 | (6,9) | 3 | (2,3) | 2 | 3 |
| 4 | (5,5) | 5 | (1,1) | 1 | 4 |
| 5 | (3,2) | 1 | (3,2) | 0 | 4 |

In [18]:
def trace_gcd(rectangles):
    cnt = defaultdict(int)
    ans = 0
    for j, (w, h) in enumerate(rectangles):
        g = gcd(w, h)
        key = (w // g, h // g)
        before = cnt[key]
        ans += before
        cnt[key] += 1
        print(f"j={j}: (w,h)=({w},{h}) g={g} khóa={key} cnt trước={before} -> ans={ans} | cnt={dict(cnt)}")
    return ans

print("Đáp án:", trace_gcd(mixed))

j=0: (w,h)=(2,3) g=1 khóa=(2, 3) cnt trước=0 -> ans=0 | cnt={(2, 3): 1}
j=1: (w,h)=(4,6) g=2 khóa=(2, 3) cnt trước=1 -> ans=1 | cnt={(2, 3): 2}
j=2: (w,h)=(1,1) g=1 khóa=(1, 1) cnt trước=0 -> ans=1 | cnt={(2, 3): 2, (1, 1): 1}
j=3: (w,h)=(6,9) g=3 khóa=(2, 3) cnt trước=2 -> ans=3 | cnt={(2, 3): 3, (1, 1): 1}
j=4: (w,h)=(5,5) g=5 khóa=(1, 1) cnt trước=1 -> ans=4 | cnt={(2, 3): 3, (1, 1): 2}
j=5: (w,h)=(3,2) g=1 khóa=(3, 2) cnt trước=0 -> ans=4 | cnt={(2, 3): 3, (1, 1): 2, (3, 2): 1}
Đáp án: 4


In [19]:
print("Đáp án:", trace_gcd(examples[0][0]))

j=0: (w,h)=(4,8) g=4 khóa=(1, 2) cnt trước=0 -> ans=0 | cnt={(1, 2): 1}
j=1: (w,h)=(3,6) g=3 khóa=(1, 2) cnt trước=1 -> ans=1 | cnt={(1, 2): 2}
j=2: (w,h)=(10,20) g=10 khóa=(1, 2) cnt trước=2 -> ans=3 | cnt={(1, 2): 3}
j=3: (w,h)=(15,30) g=15 khóa=(1, 2) cnt trước=3 -> ans=6 | cnt={(1, 2): 4}
Đáp án: 6


## 9. Code nộp LeetCode

In [20]:
class Solution:
    def interchangeableRectangles(self, rectangles: List[List[int]]) -> int:
        cnt = defaultdict(int)     # khóa tối giản -> số hình đã gặp
        ans = 0
        for w, h in rectangles:
            g = gcd(w, h)
            key = (w // g, h // g)
            ans += cnt[key]        # ghép với mọi hình cùng tỉ lệ đứng trước
            cnt[key] += 1
        return ans

for rects, exp in examples:
    assert Solution().interchangeableRectangles(rects) == exp
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

**Bổ đề (khóa duy nhất):** với `w, h ≥ 1`, `w/h = w'/h'` ⇔ `(w/g, h/g) = (w'/g', h'/g')` với `g = gcd(w,h)`, `g' = gcd(w',h')`.
- (⇐) hiển nhiên: hai phân số có cùng dạng tối giản thì bằng nhau.
- (⇒) đặt `p = w/g, q = h/g` (nguyên tố cùng nhau) và tương tự `p', q'`. Từ `p/q = p'/q'` ⇒ `p·q' = p'·q`. Vì `gcd(p, q) = 1` và `q | p·q'` ⇒ `q | q'`; đối xứng `q' | q` ⇒ `q = q'`, kéo theo `p = p'`.

**Đếm đúng:** mỗi cặp hợp lệ `(i, j)` với `i < j` được đếm **đúng một lần** — tại thời điểm xử lý `j`, khi `i` đã nằm trong `cnt[key]`. Không cặp không hợp lệ nào được đếm vì `cnt[key]` chỉ chứa các hình cùng khóa (cùng tỉ lệ theo bổ đề).

In [21]:
# Kiểm chứng bổ đề trên mọi cặp (w, h), (w', h') với giá trị 1..25
vals = [(w, h) for w in range(1, 26) for h in range(1, 26)]
key = lambda wh: (wh[0] // gcd(*wh), wh[1] // gcd(*wh))
bad = sum(1 for a in vals for b in vals if (a[0] * b[1] == a[1] * b[0]) != (key(a) == key(b)))
print("Số cặp vi phạm bổ đề:", bad, "trên", len(vals) ** 2, "cặp")

Số cặp vi phạm bổ đề: 0 trên 390625 cặp


## 11. Phân tích Time / Space

Gọi `n` = số hình, `V = 10⁵` = giá trị lớn nhất.

| Cách | Time | Space |
|---|---|---|
| Brute force | `O(n²)` | `O(1)` |
| Float key | `O(n)` trung bình | `O(n)` |
| GCD key + hash | `O(n log V)` trung bình | `O(n)` |
| GCD key + sort | `O(n log n + n log V)` | `O(n)` |
| Fraction key | `O(n log V)` nhưng hằng số lớn | `O(n)` |

"Trung bình" vì phép băm có worst case `O(n)` mỗi thao tác nếu va chạm cực đoan (gần như không xảy ra với tuple số nguyên). Đáp án có thể tới `C(10⁵, 2) ≈ 5·10⁹` — vượt `int32`; Python không lo, nhưng Java/C++ phải dùng `long`.

In [22]:
rng = random.Random(2001)
big = [[rng.randint(1, 10**5), rng.randint(1, 10**5)] for _ in range(10**5)]
big_same = [[i, 2 * i] for i in range(1, 50001)] * 2      # 10^5 hình cùng tỉ lệ 1/2
for name, fn in [("float", count_float), ("gcd", count_gcd), ("sort", count_sort), ("fraction", count_fraction)]:
    t = time.perf_counter(); r1 = fn(big); r2 = fn(big_same)
    print(f"{name:9s}: ngẫu nhiên={r1}, cùng tỉ lệ={r2}  ({time.perf_counter() - t:.3f}s)")
print("C(10^5, 2) =", comb(10**5, 2), "> 2^31 - 1 =", 2**31 - 1)

float    : ngẫu nhiên=5, cùng tỉ lệ=4999950000  (0.036s)


gcd      : ngẫu nhiên=5, cùng tỉ lệ=4999950000  (0.089s)
sort     : ngẫu nhiên=5, cùng tỉ lệ=4999950000  (0.103s)


fraction : ngẫu nhiên=5, cùng tỉ lệ=4999950000  (0.282s)
C(10^5, 2) = 4999950000 > 2^31 - 1 = 2147483647


**Vì sao khóa float vẫn đúng trong ràng buộc?** Hai phân số khác nhau `a/b ≠ c/d` có hiệu `|ad − bc|/(bd) ≥ 1/(bd) ≥ 10⁻¹⁰`. Phép chia `float` (IEEE-754 double) cho kết quả **làm tròn đúng**, sai số tương đối ≤ `2⁻⁵³ ≈ 1.1·10⁻¹⁶`; với giá trị tỉ lệ ≤ `10⁵`, sai số tuyệt đối ≤ `1.1·10⁻¹¹` < nửa khoảng cách tối thiểu. Hơn nữa phép chia làm tròn đúng là **hàm đơn điệu**, nên hai phân số bằng nhau luôn cho cùng float. Ô dưới kiểm thực nghiệm: lấy các cặp tỉ lệ "kề nhau" khó nhất (mẫu lớn) và xem float có phân biệt được không.

In [23]:
rng = random.Random(11)
collisions = 0
for _ in range(20000):
    b = rng.randint(50000, 10**5); d = rng.randint(50000, 10**5)
    a = rng.randint(1, 10**5)
    # c gần a*d/b nhất nhưng tỉ lệ khác: thử c và c +- 1
    c0 = a * d // b
    for c in (c0 - 1, c0, c0 + 1):
        if 1 <= c <= 10**5 and a * d != b * c and a / b == c / d:
            collisions += 1
print("Số lần float nhầm hai tỉ lệ khác nhau (trong ràng buộc):", collisions)

Số lần float nhầm hai tỉ lệ khác nhau (trong ràng buộc): 0


## 12. Bộ test

Gồm ví dụ đề, biên (`n = 1`, mọi hình giống nhau, hình vuông, `(a,b)` và `(b,a)`), và đối chiếu 5 cách với brute force trên dữ liệu ngẫu nhiên (seed cố định), kể cả dữ liệu giá trị nhỏ để có nhiều tỉ lệ trùng.

In [24]:
all_fns = {
    "brute": count_brute, "float": count_float, "gcd": count_gcd,
    "sort": count_sort, "fraction": count_fraction,
    "Solution": lambda r: Solution().interchangeableRectangles(r),
    "SolutionSort": lambda r: SolutionSort().interchangeableRectangles(r),
    "SolutionFloat": lambda r: SolutionFloat().interchangeableRectangles(r),
}
edge = [
    ([[1, 1]], 0),                         # n = 1
    ([[5, 5]] * 5, 10),                    # 5 hình giống hệt: C(5,2)
    ([[2, 3], [3, 2]], 0),                 # xoay không tính
    ([[1, 1], [2, 2], [100000, 100000]], 3),
    ([[99999, 100000], [99998, 99999]], 0),  # hai tỉ lệ rất gần nhau
    ([[1, 100000], [2, 100000]], 0),
]
for rects, exp in examples + edge:
    for name, fn in all_fns.items():
        assert fn(rects) == exp, (name, rects)
print("Ví dụ + biên: OK")

Ví dụ + biên: OK


In [25]:
rng = random.Random(42)
for trial in range(300):
    n = rng.randint(1, 60)
    hi = rng.choice([3, 6, 12, 100000])
    rects = [[rng.randint(1, hi), rng.randint(1, hi)] for _ in range(n)]
    want = count_brute(rects)
    for name, fn in all_fns.items():
        assert fn(rects) == want, (name, rects)
print("300 test ngẫu nhiên: mọi cách khớp brute force")

300 test ngẫu nhiên: mọi cách khớp brute force


In [26]:
# Tạo có chủ đích nhiều hình cùng tỉ lệ: nhân một tỉ lệ cơ sở với hệ số ngẫu nhiên
rng = random.Random(7)
for _ in range(100):
    bases = [(rng.randint(1, 9), rng.randint(1, 9)) for _ in range(4)]
    rects = []
    for _ in range(rng.randint(1, 80)):
        p, q = rng.choice(bases); m = rng.randint(1, 10**5 // 9)
        rects.append([p * m, q * m])
    assert count_gcd(rects) == count_brute(rects) == count_float(rects)
print("100 test 'nhiều tỉ lệ trùng': OK")

100 test 'nhiều tỉ lệ trùng': OK


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Dùng chia nguyên `w // h` làm khóa

Đề nói rõ "chia số thực". `4 // 5 == 0` và `7 // 8 == 0` nên bị coi là cùng tỉ lệ.

In [27]:
def wrong_floor_div(rectangles):
    cnt = Counter(w // h for w, h in rectangles)     # SAI
    return sum(c * (c - 1) // 2 for c in cnt.values())

r = [[4, 5], [7, 8]]
print("sai:", wrong_floor_div(r), "| đúng:", count_brute(r))

sai: 1 | đúng: 0


In [28]:
print("sửa (khóa tối giản):", count_gcd([[4, 5], [7, 8]]))

sửa (khóa tối giản): 0


### Lỗi 2 — Không tối giản, dùng thẳng `(w, h)` làm khóa

`(4, 8)` và `(3, 6)` cùng tỉ lệ nhưng là hai tuple khác nhau.

In [29]:
def wrong_no_reduce(rectangles):
    cnt = Counter((w, h) for w, h in rectangles)      # SAI: chưa chia gcd
    return sum(c * (c - 1) // 2 for c in cnt.values())

r = examples[0][0]
print("sai:", wrong_no_reduce(r), "| đúng:", count_brute(r))

sai: 0 | đúng: 6


In [30]:
print("sửa:", count_gcd(examples[0][0]))

sửa: 6


### Lỗi 3 — Đếm cặp sai công thức (`c*c` hoặc quên chia 2)

`c*c` đếm cả cặp `(i, i)` và đếm mỗi cặp hai lần; `c*(c-1)` vẫn đếm hai lần `(i,j)` và `(j,i)`.

In [31]:
def wrong_pair_formula(rectangles):
    cnt = Counter((w // gcd(w, h), h // gcd(w, h)) for w, h in rectangles)
    return sum(c * (c - 1) for c in cnt.values())    # SAI: thiếu // 2

r = examples[0][0]
print("sai:", wrong_pair_formula(r), "| đúng:", count_brute(r))

sai: 12 | đúng: 6


In [32]:
def fixed_pair_formula(rectangles):
    cnt = Counter((w // gcd(w, h), h // gcd(w, h)) for w, h in rectangles)
    return sum(c * (c - 1) // 2 for c in cnt.values())

print("sửa:", fixed_pair_formula(examples[0][0]))

sửa: 6


### Lỗi 4 — Tin tuyệt đối vào khóa float khi giá trị lớn

Trong ràng buộc của đề (`≤ 10⁵`) khóa float vẫn đúng. Nhưng thói quen này nguy hiểm: với giá trị cỡ `10¹⁷` (**ngoài ràng buộc**, chỉ để minh họa) hai tỉ lệ khác nhau cho **cùng** một float.

In [33]:
r = [[10**17 + 1, 10**17], [1, 1]]
print("float cho rằng bằng nhau:", (10**17 + 1) / 10**17 == 1 / 1)
print("sai (float):", count_float(r), "| đúng (nhân chéo):", count_brute(r))

float cho rằng bằng nhau: True
sai (float): 1 | đúng (nhân chéo): 0


In [34]:
print("sửa (gcd, chính xác):", count_gcd([[10**17 + 1, 10**17], [1, 1]]))

sửa (gcd, chính xác): 0


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Đếm số cặp (i<j) thỏa f(i) == f(j)"** ⇒ đếm tần suất theo khóa `f`, cộng `C(c,2)` hoặc `ans += cnt[key]`.
- **Khóa là tỉ lệ / hệ số góc / hướng** ⇒ tối giản bằng `gcd` (chú ý dấu nếu có số âm).
- So sánh phân số ⇒ **nhân chéo**; lưu phân số ⇒ **tuple tối giản**.
- Đáp án là số cặp ⇒ có thể tới `n²/2` ⇒ cẩn thận tràn số ở ngôn ngữ khác.
- Gặp `n = 10⁵` ⇒ gạch bỏ ngay `O(n²)`.

## 15. Biến thể của bài

1. **Cho phép xoay hình** (coi `(w,h)` và `(h,w)` cùng nhóm): khóa = tối giản của `(min, max)`.
2. **Đếm cặp điểm cùng hệ số góc với gốc tọa độ** (có số âm, số 0): chuẩn hóa dấu, xử lý mẫu bằng 0.
3. **LeetCode 149 — Max Points on a Line:** với mỗi điểm, đếm tần suất hệ số góc (tối giản) tới các điểm khác.
4. **LeetCode 1512 — Number of Good Pairs:** cùng kỹ thuật đếm cặp, khóa là chính giá trị.
5. **Đếm cặp có tích tỉ lệ bằng 1** (`w_i/h_i · w_j/h_j = 1`): tra `cnt[(h', w')]` với khóa đảo ngược.

Minh họa biến thể 1 và 5 (tự dựng, kiểm bằng brute force):

In [35]:
def count_rotate_ok(rectangles):
    cnt = Counter()
    for w, h in rectangles:
        a, b = min(w, h), max(w, h); g = gcd(a, b)
        cnt[(a // g, b // g)] += 1
    return sum(comb(c, 2) for c in cnt.values())

def count_inverse_pairs(rectangles):
    cnt = Counter(); ans = 0
    for w, h in rectangles:
        g = gcd(w, h); p, q = w // g, h // g
        ans += cnt[(q, p)]            # cần tỉ lệ nghịch đảo đã gặp
        cnt[(p, q)] += 1
    return ans

rng = random.Random(15)
for _ in range(200):
    rects = [[rng.randint(1, 8), rng.randint(1, 8)] for _ in range(rng.randint(1, 30))]
    n = len(rects)
    b1 = sum(1 for i in range(n) for j in range(i + 1, n)
             if rects[i][0] * rects[j][1] == rects[i][1] * rects[j][0] or rects[i][0] * rects[j][0] == rects[i][1] * rects[j][1])
    b5 = sum(1 for i in range(n) for j in range(i + 1, n) if rects[i][0] * rects[j][0] == rects[i][1] * rects[j][1])
    assert count_rotate_ok(rects) == b1 and count_inverse_pairs(rects) == b5
print("Biến thể 1 (cho xoay) và 5 (tích = 1): OK; ví dụ:", count_rotate_ok([[2, 3], [3, 2], [4, 6]]), count_inverse_pairs([[2, 3], [3, 2], [6, 4]]))

Biến thể 1 (cho xoay) và 5 (tích = 1): OK; ví dụ: 3 2


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | nhân chéo mọi cặp | `O(n²)` | `O(1)` | `n` nhỏ, làm chuẩn kiểm tra |
| 2. Float key | `Counter(w/h)` | `O(n)` | `O(n)` | nhanh, ngắn; chỉ khi chắc chắn đủ độ chính xác |
| 3. GCD + hash | khóa tối giản, `ans += cnt` | `O(n log V)` | `O(n)` | **nộp bài**, chính xác |
| 4. GCD + sort | sắp khóa, đếm đoạn | `O(n log n)` | `O(n)` | không muốn dùng hash |
| 5. Fraction | `Counter(Fraction)` | `O(n log V)` (chậm hơn) | `O(n)` | viết nhanh, chính xác |

## 17. Bài tập (từ dễ đến khó)

1. **Đoán output:** `[[1,2],[2,4],[3,6],[1,3]]`.
2. **Tính tay:** điền bảng trace (khóa, `cnt` trước, `ans`) cho `[[6,4],[3,2],[9,6],[2,3],[4,6]]`.
3. **Điền TODO:** hoàn thiện `reduce_ratio(w, h)` trả về khóa tối giản.
4. **Tự viết lại:** viết lại Cách 3 không nhìn code.
5. **Mini challenge:** cho danh sách điểm nguyên `(x, y)` khác `(0,0)` (có thể âm), đếm số cặp điểm nằm trên **cùng một đường thẳng qua gốc tọa độ**.

In [36]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def reduce_ratio(w, h):
    # TODO: chia cho gcd
    pass

In [37]:
# Bài 4 — TODO (chỉ định nghĩa, không gọi)
def my_count(rectangles):
    # TODO
    pass

In [38]:
# Bài 5 — TODO (chỉ định nghĩa, không gọi)
def count_same_line_through_origin(points):
    # TODO: chuẩn hóa (x, y) theo gcd và theo dấu
    pass

## 18. Gợi ý 3 mức

**Bài 1.** (1) Tối giản từng hình. (2) `(1,2)` xuất hiện 3 lần, `(1,3)` 1 lần. (3) `C(3,2) = 3`.

**Bài 2.** (1) Khóa: `(3,2),(3,2),(3,2),(2,3),(2,3)`. (2) `ans` tăng thêm `0,1,2,0,1`. (3) Tổng `4`.

**Bài 3.** (1) `g = gcd(w, h)`. (2) Trả về `tuple`. (3) `(w // g, h // g)`.

**Bài 4.** (1) `defaultdict(int)`. (2) Cộng `cnt[key]` **trước** khi tăng. (3) Trả `ans`.

**Bài 5.** (1) Hai điểm cùng đường qua gốc ⇔ `x1·y2 == x2·y1`. (2) Chia cho `gcd(|x|, |y|)`. (3) Chuẩn hóa dấu: đường thẳng chứa cả `(x,y)` và `(−x,−y)`, nên ép `x > 0`, hoặc `x == 0` thì `y > 0`.

## 19. Đáp án tham khảo

In [39]:
# Bài 1, 2
assert Solution().interchangeableRectangles([[1, 2], [2, 4], [3, 6], [1, 3]]) == 3
assert Solution().interchangeableRectangles([[6, 4], [3, 2], [9, 6], [2, 3], [4, 6]]) == 4
print("Bài 1: 3 | Bài 2: 4")
trace_gcd([[6, 4], [3, 2], [9, 6], [2, 3], [4, 6]])

Bài 1: 3 | Bài 2: 4
j=0: (w,h)=(6,4) g=2 khóa=(3, 2) cnt trước=0 -> ans=0 | cnt={(3, 2): 1}
j=1: (w,h)=(3,2) g=1 khóa=(3, 2) cnt trước=1 -> ans=1 | cnt={(3, 2): 2}
j=2: (w,h)=(9,6) g=3 khóa=(3, 2) cnt trước=2 -> ans=3 | cnt={(3, 2): 3}
j=3: (w,h)=(2,3) g=1 khóa=(2, 3) cnt trước=0 -> ans=3 | cnt={(3, 2): 3, (2, 3): 1}
j=4: (w,h)=(4,6) g=2 khóa=(2, 3) cnt trước=1 -> ans=4 | cnt={(3, 2): 3, (2, 3): 2}


4

In [40]:
# Bài 3, 4
def reduce_ratio_ans(w, h):
    g = gcd(w, h)
    return (w // g, h // g)

def my_count_ans(rectangles):
    cnt = defaultdict(int); ans = 0
    for w, h in rectangles:
        k = reduce_ratio_ans(w, h)
        ans += cnt[k]; cnt[k] += 1
    return ans

assert reduce_ratio_ans(10, 20) == (1, 2) and reduce_ratio_ans(7, 7) == (1, 1)
rng = random.Random(4)
for _ in range(200):
    rects = [[rng.randint(1, 10), rng.randint(1, 10)] for _ in range(rng.randint(1, 40))]
    assert my_count_ans(rects) == count_brute(rects)
print("Bài 3, 4: OK")

Bài 3, 4: OK


In [41]:
# Bài 5
def count_same_line_through_origin_ans(points):
    cnt = defaultdict(int); ans = 0
    for x, y in points:
        g = gcd(abs(x), abs(y))
        x, y = x // g, y // g
        if x < 0 or (x == 0 and y < 0):
            x, y = -x, -y
        ans += cnt[(x, y)]; cnt[(x, y)] += 1
    return ans

def brute_line(points):
    n = len(points)
    return sum(1 for i in range(n) for j in range(i + 1, n)
               if points[i][0] * points[j][1] == points[i][1] * points[j][0])

rng = random.Random(5)
for _ in range(300):
    pts = []
    while len(pts) < rng.randint(1, 30):
        p = (rng.randint(-6, 6), rng.randint(-6, 6))
        if p != (0, 0): pts.append(p)
    assert count_same_line_through_origin_ans(pts) == brute_line(pts)
print("Bài 5: OK; ví dụ:", count_same_line_through_origin_ans([(1, 2), (-2, -4), (0, 3), (0, -1), (2, 1)]))

Bài 5: OK; ví dụ: 2


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "đếm cặp cùng khóa":**

```python
cnt = defaultdict(int); ans = 0
for item in items:
    key = normalize(item)       # ở bài này: (w//g, h//g)
    ans += cnt[key]             # ghép với các phần tử trước
    cnt[key] += 1
return ans
```

In [42]:
def template(items, normalize):
    cnt = defaultdict(int); ans = 0
    for it in items:
        k = normalize(it); ans += cnt[k]; cnt[k] += 1
    return ans
print([template(r, lambda wh: reduce_ratio_ans(*wh)) for r, _ in examples])

[6, 0]


**Checklist:**

- [ ] Khóa là phân số **tối giản** (hoặc float nếu chắc chắn đủ chính xác), không phải `w // h`, không phải `(w, h)` thô.
- [ ] Khóa là `tuple` (hashable), không phải `list`.
- [ ] Đếm cặp `c(c−1)/2` hoặc `ans += cnt[key]` **trước** `cnt[key] += 1`.
- [ ] `(a, b)` và `(b, a)` là **khác nhau** (đề không cho xoay).
- [ ] Đáp án có thể ~`5·10⁹` (dùng `long` nếu dịch sang Java/C++).
- [ ] Đã thử `n = 1`, mọi hình giống nhau, hai tỉ lệ rất gần nhau.